In [1]:
from nichenetpy.utils import (
    read_csv_cols,
    read_csv_rows,
    extract_ligands_from_settings
)
from nichenetpy.model_construction import (
    construct_weighted_networks,
    construct_ligand_target_matrix,
    apply_hub_correction
)
from nichenetpy.evaluation import (
    convert_expression_settings_evaluation,
    convert_settings_ligand_prediction,
    get_single_ligand_importances,
    evaluate_single_importances_ligand_prediction
)
from nichenetpy.prediction import LigandActivityPredictor

from itertools import chain, repeat

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]
ligands = extract_ligands_from_settings(settings)

In [6]:
gr_network = gr_network[(gr_network["database"] == "NicheNet_LT") | (gr_network["database"] == "CytoSig")]
gr_network = gr_network[[
    not e for e in
    (
        (gr_network["database"] == "NicheNet_LT") &
        np.array([fr in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]])
    ) |
    (
        (gr_network["database"] == "CytoSig") &
        np.array([fr in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]])
    )
]]

In [7]:
source_weights = dict(zip(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])), repeat(1)))
weighted_networks = construct_weighted_networks(
    lr_network,
    sig_network,
    gr_network,
    source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)

In [8]:
predictor = LigandActivityPredictor(
    *construct_ligand_target_matrix(
        weighted_networks,
        lr_network,
        ligands,
        damping_factor=0.789,
        ltf_cutoff=0.926
    )
)

In [12]:
performances = {
    k: predictor.evaluate_target_prediction(v["from"] if type(v["from"]) is str else "-".join(v["from"]), v["response"])
    for k, v in settings.items()
}

C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic
C:\Users\victorm\Documents\nichenetpy\src\nichenetpy\metrics.py:160: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pcc = pearsonr(response, prediction).statistic


ValueError: TNF-LTB not in ligand_target_matrix

In [ ]:
session_info.show()